# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alizawwaris974/Assignment-1---Flyrank-ML/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
Finding A: "The Freshness Multiplier" (Finding #4) — the 3.2x health / 57x impressions refresh claim.
Finding: "365+ day content that was refreshed within 30 days shows 3.2x health boost (from 10.7 to 34.5) and 57x more impressions (from 71 to 4039)."

Methodology question: The paper itself models good practice here — it already flags that the 361+ freshness buckets 283:1 growth ratio is unstable (only 1 declining page).
Extending that same scrutiny one step further: is the 57x/3.2x comparison a matched cohort (refreshed pages vs. otherwise-similar non-refreshed pages, controlling for the fact
that editors likely chose to refresh pages with pre-existing signals worth saving), or a before/after on the same pages? If it is the latter, some of the lift could reflect regression
towards the mean (a pages worst 30-day window naturally recovering somewhat) or selection bias (editors refreshing pages they already suspected still had value) rather than the
refresh causing the full effect. The paper doesnt state the sample size behind this specific 57x figure either — worth knowing, given the adjacent 361+ bucket was explicitly called
out as too small to trust.

Finding B: The ML appendixs growth-prediction Logistic Regression (71% holdout accuracy).
Finding: "Logistic regression (71% holdout accuracy) describing which sampled features separate growing from declining pages."

Methodology question: Two things worth asking, both drawn directly from this same capstones own hard-won lessons. First, whats the base rate?
Finding #1 reports 74.8K growing vs. 45.6K declining pages elsewhere in the paper — that implies roughly a 62%/38% split, meaning a naive "always predict growing"
rule would already score ~62% accuracy. If so, 71% represents only about 9 points of real skill, not 71 — the same gap the field itself warns readers to check for.
Second, was the 80/20 holdout split grouped by brand, or random by row? With 57 brands in the portfolio (structurally similar to this capstones client_hash_id),
a random split risks letting the model partly learn brand-specific writing style or template rather than a generalizable growth signal — inflating the reported accuracy
in a way that would not hold for a genuinely new brand. The methodology page does not specify which was used.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import duckdb, getpass
import pandas as pd, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ').strip()
conn = duckdb.connect()
conn.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

# Same feature build as w05
momentum_query = f"""
    WITH feb AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_prev
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    ),
    mar AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_last,
               AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS avg_position_mar
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_impressions IS NOT NULL
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT mar.*, feb.imp_prev
    FROM mar LEFT JOIN feb
      ON mar.client_hash_id = feb.client_hash_id AND mar.content_hash_id = feb.content_hash_id
    WHERE mar.imp_last IS NOT NULL AND feb.imp_prev >= 100
"""
df = conn.execute(momentum_query).df()
content_df = conn.execute(f"""
    SELECT content_hash_id, content_type, word_count, content_created_date
    FROM read_parquet('{REL}/dim_content.parquet')
    WHERE is_published IS TRUE AND is_deleted IS FALSE
""").df()
df = df.merge(content_df, on="content_hash_id", how="inner")
df["is_declining"] = (df["imp_last"] < 0.8 * df["imp_prev"]).astype(int)
df["content_age_days"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(df["content_created_date"])).dt.days
df["word_count"] = df["word_count"].fillna(df["word_count"].median())

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

feature_cols = ["avg_position_mar", "imp_prev", "word_count"]  # content_age_days dropped, per w05's own MIXED-signal finding

def run_split(df, grouped, seed=42):
    np.random.seed(seed)
    if grouped:
        clients = df["client_hash_id"].unique()
        np.random.shuffle(clients)
        test_clients = set(clients[:max(1, int(len(clients) * 0.2))])
        test_mask = df["client_hash_id"].isin(test_clients)
    else:
        test_mask = pd.Series(np.random.rand(len(df)) < 0.2, index=df.index)  # random row split — the DISHONEST version
    tr, te = df[~test_mask], df[test_mask]
    Xtr, Xte = tr[feature_cols].fillna(0), te[feature_cols].fillna(0)
    ytr, yte = tr["is_declining"].values, te["is_declining"].values
    rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42).fit(Xtr, ytr)
    proba = rf.predict_proba(Xte)[:, 1]
    return {
        "split_type": "grouped (honest)" if grouped else "random (dishonest)",
        "test_n": len(te),
        "auc": roc_auc_score(yte, proba),
        "precision_at_10": precision_at_k(proba, yte, 10)
    }

before_after = pd.DataFrame([run_split(df, grouped=False), run_split(df, grouped=True)])
print(before_after.to_string(index=False))

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

        split_type  test_n      auc  precision_at_10
random (dishonest)   15288 0.663777              1.0
  grouped (honest)   20822 0.558466              1.0


Results show a random-split AUC looking better than grouped (since row-level random split leaks client-level patterns into training), and the size of that gap is itself the finding.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
X_honest = df[feature_cols].fillna(0)
y = df["is_declining"].values

clients = df["client_hash_id"].unique()
np.random.seed(42); np.random.shuffle(clients)
test_clients = set(clients[:max(1, int(len(clients) * 0.2))])
test_mask = df["client_hash_id"].isin(test_clients)

rf_honest = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42).fit(X_honest[~test_mask], y[~test_mask])
honest_auc = roc_auc_score(y[test_mask], rf_honest.predict_proba(X_honest[test_mask])[:, 1])
print(f"HONEST AUC (no label-derived features): {honest_auc:.4f}")

# Deliberately add imp_last (part of the label's own numerator) — the leak
X_leak = X_honest.copy()
X_leak["imp_last"] = df["imp_last"].values
rf_leak = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42).fit(X_leak[~test_mask], y[~test_mask])
leak_auc = roc_auc_score(y[test_mask], rf_leak.predict_proba(X_leak[test_mask])[:, 1])
print(f"LEAKED AUC (with imp_last added): {leak_auc:.4f}")
print(f"Jump: {leak_auc - honest_auc:.4f} — confirms the test harness catches leakage when it's really there.")

HONEST AUC (no label-derived features): 0.5585
LEAKED AUC (with imp_last added): 0.9781
Jump: 0.4196 — confirms the test harness catches leakage when it's really there.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In repeated client-grouped holdout splits, Random Forest showed a directional, decision-support improvement over the baseline rule at Precision@10 —
consistently ranking higher across 5 of 5 random splits, though the magnitude varied considerably (0.40-1.00) given the small number of available test clients.
 This is an observed pattern in this dataset, not a guarantee of performance on a new client or a claim that the model has found a causal driver of decline."

## Self-check

Before you submit, confirm each line honestly:

- [T] Every section above is filled — markdown thinking AND the code that backs it
- [T] The notebook runs top to bottom with no errors (Runtime → Run all)
- [T] No client names, URLs, or private queries anywhere
- [T] My claims use careful words: observed, measured, directional, decision-support
- [T] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.